# Capstone · Phase 3：Agentic系统架构设计 · LangGraph多Agent营销系统

**版本**：v5.0 学习材料包
**配套**：[notes.md](./notes.md)（讲义）｜ [data/README.md](./data/README.md)（真实库说明）｜ [reading.md](./reading.md)（深链阅读）
**核心命题**：如何用LangGraph的有状态图构建Capstone三层架构的Agent编排系统，整合Phase 2知识图谱数据层，实现researcher->strategist->writer多Agent协作 + 条件分支 + 人机协同审批（HITL）？
**v5.0升级**：真实LangGraph库上机（非伪代码）+ TODO填空式起始笔记本 + 离线模拟LLM fallback + 整合技能5(Agent架构)+技能2(编排)
**整合性**：本Phase调用Phase 2知识图谱作为Agent知识基础，产出的Agent系统为Phase 4因果评估提供决策Agent

## 0. 环境准备

首次运行需安装依赖（取消注释执行一次）：
- `langgraph`：状态图编排框架（StateGraph / 条件边 / 检查点 / interrupt）
- `langchain-openai` + `langchain-core`：LLM调用与消息类型
- `pydantic`：状态schema验证

无 `OPENAI_API_KEY` 时自动降级为**离线模拟LLM**（返回固定营销文案），图可端到端跑通。编排逻辑（StateGraph/条件边/interrupt/Checkpointing）全部是真实LangGraph API。

In [ ]:
# === 环境准备 ===
# 首次运行取消注释：!pip install langgraph langchain-openai langchain-core pydantic -q

import operator
from typing import TypedDict, Literal, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# --- LLM 初始化 ---
# 优先使用真实 OpenAI API；无 API key 时自动降级为离线模拟 LLM
try:
    import os
    from langchain_openai import ChatOpenAI
    if os.environ.get("OPENAI_API_KEY"):
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
        LLM_MODE = "openai"
    else:
        raise ImportError("No OPENAI_API_KEY")
except Exception:
    LLM_MODE = "offline"

    class _MockResponse:
        """模拟 LLM 响应，兼容 LangChain .content 接口"""
        def __init__(self, content):
            self.content = content

    class OfflineMockLLM:
        """离线模拟 LLM：无 API key 时返回固定营销文案，使图端到端跑通。
        Capstone视角：每个分支模拟一个专用 Prompt+模型节点。
        整合技能5(Agent架构) + 技能2(编排)。"""
        _RESEARCH = (
            "【市场研究报告】\n"
            "基于Phase 2知识图谱分析：\n"
            "目标人群：25-35岁都市白领，关注健康管理，亚健康焦虑明显\n"
            "竞品：小米手环(极致性价比) / 华为Band(专业运动) / Apple Watch(高端生态)\n"
            "趋势：2026年智能穿戴趋向'全天候健康监测+AI洞察'\n"
            "知识图谱关联：心率监测<-运动健康<-都市白领需求<-健康焦虑"
        )
        _STRATEGY = (
            "【营销策略方案】\n"
            "核心主张：AI健康洞察，24小时主动守护\n"
            "差异化定位：全科健康监测+AI个性化建议，入门级智能手环首选\n"
            "渠道组合：小红书35% + 抖音30% + 京东25% + 线下10%\n"
            "信息层级：全天候监测 -> AI洞察 -> 个性化建议\n"
            "预算分配：KOL种草40% + 信息流30% + 站内20% + 品牌号10%\n"
            "Plan-Execute：先建健康认知(Plan) -> 再促转化(Execute)"
        )
        _CONTENT = (
            "【创意方案A】标题：腕上AI健康管家\n"
            "正文：心率、血氧、睡眠全天候监测，AI生成个性化健康报告。"
            "不只是手环，是你的腕上健康管家。50米防水，7天续航。\n"
            "CTA：限时首发价299元，点击体验\n"
            "渠道：小红书\n\n"
            "【创意方案B】标题：懂你的每一次心跳\n"
            "正文：24小时心率监测+血氧检测+睡眠分析，AI洞察你的健康趋势。\n"
            "CTA：抢先试用，享首发优惠\n"
            "渠道：抖音"
        )

        def invoke(self, messages):
            text = ""
            if isinstance(messages, str):
                text = messages
            elif isinstance(messages, list):
                text = " ".join(getattr(m, "content", str(m)) for m in messages)
            else:
                text = str(messages)
            if "研究" in text or "人群" in text or "分析" in text or "知识图谱" in text:
                return _MockResponse(self._RESEARCH)
            if "策略" in text or "主张" in text or "预算" in text:
                return _MockResponse(self._STRATEGY)
            return _MockResponse(self._CONTENT)

    llm = OfflineMockLLM()

print(f"LLM 模式: {LLM_MODE}")
print("LangGraph + langchain_core + pydantic 就绪")

## 1. Capstone三层架构与Phase 2整合

**Capstone三层架构**（引用独立教材Phase 3节）：
```
┌─────────────────────────────────────────────┐
│           用户交互层                          │
│  ├─ 营销人员界面（Brief输入/最终方案输出）      │
│  └─ 管理层审批（HITL人机协同）                 │
├─────────────────────────────────────────────┤
│        Agent编排层（LangGraph）               │
│  ┌──────────┐ ┌──────────┐ ┌──────────┐    │
│  │researcher │→│strategist │→│  writer  │    │
│  │(研究Agent)│ │(策略Agent)│ │(写作Agent)│    │
│  └──────────┘ └──────────┘ └──────────┘    │
│       │                            │          │
│       └──────────┬─────────────────┘          │
│                  ▼                            │
│          ┌──────────────┐                     │
│          │review(Coordinator)│                │
│          │ 安全检查+审批路由   │                │
│          └──────────────┘                     │
├─────────────────────────────────────────────┤
│         数据与知识层                          │
│  ├─ 知识图谱（Phase 2产出）                   │
│  ├─ MCP工具接入（知识检索）                   │
│  └─ A2A Agent协作（State共享）                │
└─────────────────────────────────────────────┘
```

**Phase 2整合**：researcher_agent读取Phase 2产出的知识图谱作为市场研究的知识基础（MCP工具接入概念：Agent通过标准协议调用知识图谱工具）。

**整合技能5(Agent架构Day1-2) + 技能2(编排Day2)**：
- 技能5的Agent架构模式（ReAct/Plan-Execute/Reflection）-> 本Phase用Plan-Execute
- 技能2的编排模式（顺序/条件/循环/HITL）-> 本Phase全部覆盖

## TODO 1：定义 AgentSystemState（Capstone三层架构全局状态）

**State驱动设计**：所有节点共享全局State，每个节点只更新自己负责的字段。
用 `TypedDict` 定义（LangGraph推荐），`messages` 字段用 `Annotated[list, operator.add]` 实现追加模式。

需要包含（10个字段）：
- `brief`：营销Brief输入（用户交互层）
- `knowledge_context`：Phase 2知识图谱上下文（数据与知识层）
- `market_research`：市场研究（researcher_agent写）
- `strategy`：营销策略（strategist_agent写，Plan阶段）
- `content`：创意文案（writer_agent写，Execute阶段）
- `approved`：审批决定（review_node写）
- `review_feedback`：审核反馈
- `revision_count`：修改次数（循环退出用）
- `final_output`：最终输出（publish_node写）
- `messages`：执行日志（追加模式）

In [ ]:
# 1. 定义 AgentSystemState（Capstone三层架构全局状态）
class AgentSystemState(TypedDict):
    """Capstone Agent系统状态 - 三层架构的全局状态

    三层架构映射：
    - 用户交互层：brief（输入）/ final_output（输出）
    - Agent编排层：market_research / strategy / content / approved
    - 数据与知识层：knowledge_context（Phase 2知识图谱）
    """
    brief: str                              # 营销Brief输入（用户交互层）
    knowledge_context: str                  # Phase 2知识图谱上下文（数据与知识层）
    market_research: str                    # 市场研究（researcher_agent写）
    strategy: str                           # 营销策略（strategist_agent写，Plan阶段）
    content: str                            # 创意文案（writer_agent写，Execute阶段）
    approved: bool                          # 审批决定（review_node写）
    review_feedback: str                    # 审核反馈（review_node写）
    revision_count: int                     # 修改次数（循环退出用）
    final_output: str                       # 最终输出（publish_node写）
    messages: Annotated[list, operator.add] # 执行日志（追加模式）

print("AgentSystemState 定义完成（10 个字段）")

## TODO 2：实现 researcher_agent 和 strategist_agent（顺序编排 + Plan-Execute）

**节点模式**：`def agent(state: AgentSystemState) -> dict` -- 读State、调LLM、返回State更新。

- `researcher_agent`：基于Phase 2知识图谱分析市场（MCP工具调用概念：Agent通过知识图谱检索市场知识）
- `strategist_agent`：制定营销策略（**Plan-Execute模式**的Plan阶段 -- 先规划再执行）

用 `llm.invoke([SystemMessage(...), HumanMessage(...)])` 调LLM。

In [ ]:
# 2. researcher_agent 和 strategist_agent（顺序编排 + Plan-Execute 的 Plan 阶段）
def researcher_agent(state: AgentSystemState) -> dict:
    """研究 Agent：基于 Phase 2 知识图谱分析市场（MCP 工具调用概念）

    数据与知识层集成：读取 Phase 2 产出的知识图谱作为市场研究基础。
    MCP概念：Agent 通过标准协议调用知识图谱工具检索市场知识。
    """
    kg_context = state.get("knowledge_context", "")
    response = llm.invoke([
        SystemMessage(content=(
            "你是资深市场研究分析师。基于知识图谱上下文，分析营销 Brief，输出：\n"
            "1. 目标人群画像（年龄/性别/兴趣/消费习惯）\n"
            "2. 竞品分析（3个主要竞品定位与差异化）\n"
            "3. 市场趋势（2026年该品类关键趋势）\n"
            "用结构化格式输出。"
        )),
        HumanMessage(content=(
            f"营销 Brief：\n{state['brief']}\n\n"
            f"知识图谱上下文（Phase 2 产出）：\n{kg_context}"
        ))
    ])
    return {
        "market_research": response.content,
        "messages": ["[researcher] 基于知识图谱的市场研究完成"]
    }

def strategist_agent(state: AgentSystemState) -> dict:
    """策略 Agent：制定营销策略（Plan-Execute 的 Plan 阶段）

    Plan-Execute模式：先规划（Plan）再执行（Execute）。
    本节点是Plan阶段，writer_agent是Execute阶段。
    """
    response = llm.invoke([
        SystemMessage(content=(
            "你是营销策略总监。基于市场研究，制定营销策略：\n"
            "1. 核心营销主张（一句话）\n"
            "2. 差异化定位策略\n"
            "3. 渠道组合建议（至少3个渠道）\n"
            "4. 信息层级（主信息->支撑->证据）\n"
            "5. 预算分配建议（百分比）"
        )),
        HumanMessage(content=f"市场研究：\n{state.get('market_research', '')}")
    ])
    return {
        "strategy": response.content,
        "messages": ["[strategist] 营销策略制定完成（Plan阶段）"]
    }

print("researcher_agent + strategist_agent 定义完成")

## TODO 3：实现 writer_agent（Plan-Execute执行阶段 + 循环修改）

**Plan-Execute的Execute阶段**：基于策略生成文案。若有审核反馈（`review_feedback`），针对性修改。

关键：检查 `state.get("review_feedback")` 是否存在，若有则加入Prompt让LLM据此改进。

In [ ]:
# 3. writer_agent（Plan-Execute 执行阶段 + 循环修改）
def writer_agent(state: AgentSystemState) -> dict:
    """写作 Agent：基于策略生成文案（Plan-Execute 的 Execute 阶段 + 循环修改）

    Execute阶段：基于Plan阶段的策略生成创意文案。
    若有审核反馈（review_feedback），针对性修改。
    """
    feedback = state.get("review_feedback", "")
    revision_count = state.get("revision_count", 0)

    system_prompt = (
        "你是创意文案总监。基于营销策略生成2套创意方案，\n"
        "每套含：标题(10字内) + 正文(100-200字) + CTA + 渠道。"
    )
    if revision_count > 0 and feedback:
        system_prompt += (
            f"\n\n这是第{revision_count + 1}次修改。"
            f"上次审核反馈：{feedback}\n请据此改进。"
        )

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"营销策略：\n{state.get('strategy', '')}")
    ])
    return {
        "content": response.content,
        "messages": [f"[writer] 第{revision_count + 1}次生成文案（Execute阶段）"]
    }

print("writer_agent 定义完成")

## 2. 条件路由与循环退出（Coordinator角色）

**条件路由**是LangGraph实现分支与循环的关键：
1. **条件函数**：`def route(state) -> Literal["publish", "revise"]` -- 读State决定下一节点
2. **条件边注册**：`workflow.add_conditional_edges("review", route, {"publish": "publish", "revise": "writer"})`

**循环退出条件（必须）**：`revision_count >= 3` 时强制发布，防止无限循环。

**Coordinator概念**（Capstone教材）：条件路由函数扮演Coordinator角色，拥有全局视图，决定下一步去哪个节点。这对应教材三层架构中的协调Agent。

## TODO 4：实现 review_node 和 route_after_review（安全检查 + 条件路由）

- `review_node`：安全检查 + 合规审核节点（整合Capstone教材的安全检查Agent概念）
  - HITL模式：人工通过 `update_state` 注入 `approved=True`，此节点记录并放行
  - 自动模式：用 `revision_count` 模拟"首次需修改、修改后通过"
  - 安全检查：检测违规用语（如"虚假"、"第一"等绝对化用语）
- `route_after_review`：条件路由函数（Coordinator/Supervisor角色），返回 `"publish"` 或 `"revise"`
  - 循环退出：`revision_count >= 3`

In [ ]:
# 4. review_node 和 route_after_review（安全检查 + 条件路由 Coordinator）
def review_node(state: AgentSystemState) -> dict:
    """审核节点：安全检查 + 合规审核（整合 Capstone 安全检查 Agent 概念）

    Capstone三层架构中的安全检查Agent：
    - 内容安全检查（检测违规用语）
    - 品牌合规性验证
    - 审批决策（HITL或自动）
    """
    rc = state.get("revision_count", 0)
    already_approved = state.get("approved", False)
    content = state.get("content", "")

    # 安全检查（安全检查Agent概念）
    safety_issues = []
    if "虚假" in content or "第一" in content:
        safety_issues.append("内容可能包含违规用语")
    if "限时" in content and rc == 0:
        safety_issues.append("促销用语需法务确认")

    if already_approved:
        # HITL：人工已通过 update_state 注入审批结果
        return {
            "revision_count": rc + 1,
            "messages": [f"[review] 人工审核通过 (第{rc + 1}次)"]
        }
    elif rc >= 1:
        # 自动模式：第二次审核通过
        return {
            "approved": True,
            "review_feedback": "修改后通过，符合品牌调性与安全规范",
            "revision_count": rc + 1,
            "messages": [f"[review] 安全审核通过 (第{rc + 1}次)"]
        }
    else:
        # 自动模式：首次审核不通过
        feedback = "；".join(safety_issues) if safety_issues else "标题情感冲击力不足，CTA需更具体"
        return {
            "approved": False,
            "review_feedback": feedback,
            "revision_count": rc + 1,
            "messages": [f"[review] 需修改 (第{rc + 1}次)：{feedback}"]
        }

def route_after_review(state: AgentSystemState) -> Literal["publish", "revise"]:
    """条件路由（Coordinator/Supervisor角色）：根据审核结果决定发布或修改

    A2A概念：Agent间通过State共享通信（同进程内A2A协作）。
    Coordinator拥有全局视图，决定下一步去哪个节点。
    """
    if state.get("revision_count", 0) >= 3:
        return "publish"  # 循环退出：超过3次强制发布
    if state.get("approved", False):
        return "publish"
    return "revise"

print("review_node + route_after_review 定义完成")

## 3. 发布节点（给定，无需填写）

`publish_node` 是工作流终点，把所有Agent输出汇总成最终方案。对应Capstone三层架构的用户交互层输出。

In [ ]:
def publish_node(state: AgentSystemState) -> dict:
    """发布节点：汇总所有Agent输出，生成最终方案（用户交互层输出）"""
    final_output = (
        "=" * 60 + "\n"
        "   Capstone Agent系统 - 营销方案最终输出\n" +
        "=" * 60 + "\n\n"
        f"【营销Brief】\n{state['brief']}\n\n"
        f"【知识图谱上下文（Phase 2）】\n{state.get('knowledge_context', '')[:200]}...\n\n"
        f"【市场研究】\n{state.get('market_research', '')}\n\n"
        f"【营销策略】\n{state.get('strategy', '')}\n\n"
        f"【创意文案】\n{state.get('content', '')}\n\n"
        f"【审核记录】修改次数:{state.get('revision_count', 0)} "
        f"最终:{'通过' if state.get('approved') else '强制发布'}\n" +
        "=" * 60
    )
    return {
        "final_output": final_output,
        "messages": [f"[publish] 方案已发布 (修改次数={state.get('revision_count', 0)})"]
    }

print("publish_node 定义完成")

## TODO 5：实现 build_agent_system（三层架构装配 + HITL + 持久化）

**StateGraph装配全流程**：
1. `StateGraph(AgentSystemState)` 创建图
2. `add_node` 添加5个节点（researcher/strategist/writer/review/publish）
3. `add_edge(START, "researcher")` + 顺序边 researcher->strategist->writer->review
4. `add_conditional_edges("review", route, {...})` 条件边
5. `add_edge("publish", END)` 终止边
6. `compile(checkpointer=MemorySaver(), interrupt_before=["review"])` 编译

**整合技能5(Agent架构) + 技能2(编排)**：图的拓扑整合了技能5的Plan-Execute模式 + 技能2的顺序/条件/循环/HITL四种编排。

In [ ]:
# 5. build_agent_system（三层架构装配 + HITL + MemorySaver 持久化）
def build_agent_system(use_hitl: bool = True):
    """构建 Capstone Agent 系统架构（三层架构 + LangGraph 编排）

    整合技能5(Agent架构) + 技能2(编排)：
    - 顺序编排：researcher -> strategist -> writer -> review
    - 条件分支 + 循环：review -> publish 或 -> writer（修订）
    - HITL（use_hitl=True）：interrupt_before=['review']
    - 状态持久化：MemorySaver 检查点

    Plan-Execute模式：strategist(Plan) + writer(Execute)
    """
    workflow = StateGraph(AgentSystemState)

    # 添加节点（Agent编排层）
    workflow.add_node("researcher", researcher_agent)
    workflow.add_node("strategist", strategist_agent)
    workflow.add_node("writer", writer_agent)
    workflow.add_node("review", review_node)
    workflow.add_node("publish", publish_node)

    # 顺序边
    workflow.add_edge(START, "researcher")
    workflow.add_edge("researcher", "strategist")
    workflow.add_edge("strategist", "writer")
    workflow.add_edge("writer", "review")

    # 条件边（Coordinator路由）
    workflow.add_conditional_edges(
        "review",
        route_after_review,
        {"publish": "publish", "revise": "writer"}
    )

    # 终止边
    workflow.add_edge("publish", END)

    # 编译：MemorySaver 检查点 + 可选 HITL
    checkpointer = MemorySaver()
    if use_hitl:
        graph = workflow.compile(
            checkpointer=checkpointer,
            interrupt_before=["review"]
        )
    else:
        graph = workflow.compile(checkpointer=checkpointer)
    return graph

# 测试图编译
try:
    _test = build_agent_system(use_hitl=False)
    print("图编译成功！节点:", list(_test.get_graph().nodes.keys()))
except Exception as e:
    print(f"图编译失败：{e}")

## 4. 基础运行：修订循环演示（给定，无需填写）

`run_basic` 构建不带interrupt的图，端到端执行。
review_node逻辑：首次审核不通过（rc=0 -> reject），修改后通过（rc=1 -> approve）。
演示"研究 -> 策略 -> 生成 -> 审核 -> 修改 -> 再审核 -> 发布"的完整循环。

**Phase 2整合**：传入知识图谱上下文作为researcher_agent的知识基础。

In [ ]:
def run_basic(brief: str, knowledge_context: str):
    """基础运行：不带 interrupt，端到端执行，演示修订循环"""
    graph = build_agent_system(use_hitl=False)
    config = {"configurable": {"thread_id": "capstone_basic_001"}}
    initial_state = {
        "brief": brief,
        "knowledge_context": knowledge_context,
        "market_research": "", "strategy": "", "content": "",
        "approved": False, "review_feedback": "",
        "revision_count": 0, "final_output": "", "messages": [],
    }
    print("=" * 60)
    print("基础运行（无HITL，演示修订循环）")
    print("=" * 60)
    result = graph.invoke(initial_state, config=config)
    print("\n--- 节点执行顺序（从messages追踪）---")
    for i, msg in enumerate(result.get("messages", [])):
        print(f"  {i+1}. {msg}")
    print(f"\n--- 修订循环结果 ---")
    print(f"修改次数: {result.get('revision_count', 0)}")
    print(f"最终审批: {'通过' if result.get('approved') else '未通过'}")
    print(f"\n--- 最终输出（前200字）---")
    print(result.get("final_output", "")[:200] + "...")
    return result

brief = """
品牌：智动（Zhidong）
产品：智能运动手环 Pro（心率+血氧+睡眠+50米防水）
目标：新品上市推广，3个月内品牌知名度提升25%
预算：80万元 | 渠道：小红书、抖音、京东
"""

knowledge_context = """
Phase 2知识图谱产出（实体->关系->实体）：
(智能运动手环)-[属于]->(可穿戴设备)
(智能运动手环)-[功能]->(心率监测)
(智能运动手环)-[功能]->(血氧检测)
(智能运动手环)-[功能]->(睡眠分析)
(心率监测)-[需求场景]->(运动健康管理)
(睡眠分析)-[需求场景]->(都市白领健康焦虑)
(竞品A)-[定位]->(极致性价比)
(竞品B)-[定位]->(专业运动)
(目标人群)-[特征]->(25-35岁都市白领)
(目标人群)-[痛点]->(亚健康焦虑)
(目标人群)-[渠道偏好]->(小红书种草)
"""

try:
    result_basic = run_basic(brief, knowledge_context)
except NotImplementedError:
    print("请先完成 TODO 1-5，再运行基础版")

## 5. 人机协同（HITL）：interrupt_before三步模式

Capstone治理的关键能力：工作流在审核节点前**暂停**，State被MemorySaver持久化；
人工审核后通过 `update_state` 注入决策，图从检查点**恢复**执行。

**三步模式**：
1. `graph.invoke(initial_state, config)` -> 执行到review前暂停
2. `graph.update_state(config, {"approved": True})` -> 注入人工决策
3. `graph.invoke(None, config)` -> 恢复执行至完成

**Capstone治理意义**：高风险营销决策设置人机协同，确保人类保留最终决策权。为Phase 4因果评估提供可审计的Agent决策链。

## TODO 6：实现 run_capstone_hitl（三步HITL运行 + 真实输出）

实现三步HITL运行：
1. 构建 `use_hitl=True` 的图
2. Step 1：invoke -> 暂停前打印已执行节点 + 暂停位置
3. Step 2：update_state 注入 `approved=True`
4. Step 3：invoke(None) 恢复 -> 打印最终输出 + 执行日志

In [ ]:
# 6. run_capstone_hitl（三步 HITL 运行 + 真实输出）
def run_capstone_hitl(brief: str, knowledge_context: str):
    """三步 HITL 运行：invoke -> update_state -> resume

    Capstone整合：Phase 2知识图谱 -> Agent系统 -> Phase 4因果评估
    HITL确保高风险营销决策保留人工审批权。
    """
    graph = build_agent_system(use_hitl=True)
    config = {"configurable": {"thread_id": "capstone_hitl_001"}}
    initial_state = {
        "brief": brief,
        "knowledge_context": knowledge_context,
        "market_research": "", "strategy": "", "content": "",
        "approved": False, "review_feedback": "",
        "revision_count": 0, "final_output": "", "messages": [],
    }

    # Step 1: 执行到审核节点前暂停（interrupt_before）
    print("=" * 60)
    print("Step 1: 执行至审核节点前暂停（interrupt_before）")
    print("=" * 60)
    state1 = graph.invoke(initial_state, config=config)
    print("已执行节点：")
    for i, msg in enumerate(state1.get("messages", [])):
        print(f"  {i+1}. {msg}")
    pause_at = graph.get_state(config).next
    print(f"暂停位置（next）：{pause_at}")
    print(f"已生成文案（前100字）：{state1.get('content', '')[:100]}...")

    # Step 2: 模拟人工审批 -> 注入 approved=True
    print("\n" + "=" * 60)
    print("Step 2: 人工审批 -> 通过（update_state 注入）")
    print("=" * 60)
    graph.update_state(config, {
        "approved": True,
        "review_feedback": "人工审核通过，符合品牌调性与安全规范"
    })
    state2 = graph.get_state(config)
    print(f"已注入：approved={state2.values.get('approved')}")
    print(f"当前 next：{state2.next}")

    # Step 3: 恢复执行至完成
    print("\n" + "=" * 60)
    print("Step 3: 恢复执行（invoke(None) resume）")
    print("=" * 60)
    final_state = graph.invoke(None, config=config)
    print(f"\n--- 完整执行日志 ---")
    for i, msg in enumerate(final_state.get("messages", [])):
        print(f"  {i+1}. {msg}")
    print(f"\n--- 最终输出（前300字）---")
    print(final_state.get("final_output", "")[:300] + "...")
    return final_state

# 运行 HITL 版
brief = """
品牌：智动（Zhidong）
产品：智能运动手环 Pro（心率+血氧+睡眠+50米防水）
目标：新品上市推广，3个月内品牌知名度提升25%
预算：80万元 | 渠道：小红书、抖音、京东
"""

knowledge_context = """
Phase 2知识图谱产出（实体->关系->实体）：
(智能运动手环)-[属于]->(可穿戴设备)
(智能运动手环)-[功能]->(心率监测)
(智能运动手环)-[功能]->(血氧检测)
(智能运动手环)-[功能]->(睡眠分析)
(心率监测)-[需求场景]->(运动健康管理)
(睡眠分析)-[需求场景]->(都市白领健康焦虑)
(竞品A)-[定位]->(极致性价比)
(竞品B)-[定位]->(专业运动)
(目标人群)-[特征]->(25-35岁都市白领)
(目标人群)-[痛点]->(亚健康焦虑)
(目标人群)-[渠道偏好]->(小红书种草)
"""

try:
    result_hitl = run_capstone_hitl(brief, knowledge_context)
except NotImplementedError:
    print("请先完成 TODO 1-5")

## 6. 反思与前沿

### 反思问题
1. 修订循环的退出条件（`revision_count >= 3`）设成多少合理？为什么？取消退出条件会怎样？
2. HITL审批节点在Capstone治理中的意义是什么？与Phase 4因果评估如何衔接？
3. Phase 2知识图谱如何提升researcher_agent的研究质量？MCP工具接入的概念价值是什么？

### 2026前沿
- **LangGraph企业编排**：LangChain团队官方Agent编排框架，生产级复杂Agent事实标准
- **MCP工具接入**：Anthropic提出的Model Context Protocol，Agent通过标准协议调用外部工具（如知识图谱检索）
- **A2A Agent协作**：Google提出的Agent-to-Agent Protocol，Agent间标准通信协议。与MCP互补：MCP接工具，A2A接Agent
- **天道推演 x 多Agent仿真**：LangGraph多Agent编排与天道推演系统同构 -- 条件边=决策分支，Checkpointing=推演假设记录，反馈学习节点=因果模型更新
- **Plan-Execute模式**：先规划(strategist)再执行(writer)的两阶段架构，比"让LLM一次性完成"更可控

### Capstone整合性
- **Phase 2 -> Phase 3**：知识图谱数据层 -> Agent编排层（本Phase）
- **Phase 3 -> Phase 4**：Agent系统 -> 因果评估（Agent决策的因果效应验证）
- **技能5 + 技能2**：Agent架构模式 + 编排模式 -> 完整的Agent系统架构